In [ ]:
def main(datasources, start_date, end_date):
    """PIT-safe, style-aware Ridge + shallow-XGBoost daily stock factor."""
    import gc

    import numpy as np
    import pandas as pd
    import dai
    from xgboost import XGBRegressor

    # The competition runner replaces start_date/end_date for each evaluation
    # slice.  Training must therefore be completely independent of both
    # arguments: otherwise changing the requested prediction window changes
    # the fitted model and the platform's prefix-invariance check flags a
    # future/data-leakage dependency.  The model must also stop before the
    # first factor date checked by the platform: training through 2024 while
    # generating 2024 factors would leak later-2024 realised returns.
    official_history_start = pd.Timestamp("2019-01-01")
    first_factor_date = pd.Timestamp("2024-01-01")
    train_end_ts = first_factor_date - pd.Timedelta(days=1)

    train_start = official_history_start.strftime("%Y-%m-%d")
    train_end = train_end_ts.strftime("%Y-%m-%d")
    train_bar_table = "bigalpha_2026_stock_bar1m"
    train_financial_table = "bigalpha_2026_financial"
    train_rows_per_date = 500
    training_chunk_days = 366

    rank_input_columns = [
        "gap",
        "intraday_return",
        "close_position",
        "short_reversal_3",
        "momentum_120_excl_20",
        "trend_efficiency_120_excl_20",
        "volatility_20",
        "downside_volatility_20",
        "volatility_ratio_20_60",
        "beta_60",
        "residual_volatility_20",
        "log_amount_20",
        "amount_surprise_20",
        "amihud_20",
        "late_book_imbalance",
        "book_imbalance_change",
        "relative_spread",
        "roa_proxy",
        "net_margin",
        "asset_turnover",
        "leverage",
    ]
    model_columns = [
        "gap",
        "intraday_return",
        "close_position",
        "short_reversal_3",
        "momentum_120_excl_20",
        "trend_efficiency_120_excl_20",
        "amount_surprise_20",
        "amihud_20",
        "volatility_ratio_20_60",
        "downside_volatility_20",
        "book_flow_quality",
        "book_imbalance_change",
        "quality_score",
        "confirmed_reversal",
        "low_vol_momentum",
        "quality_momentum",
    ]
    style_control_columns = [
        "beta_60",
        "momentum_120_excl_20",
        "residual_volatility_20",
        "log_amount_20",
        "leverage",
        "quality_score",
    ]

    def _normalise_date(frame):
        frame = frame.copy()
        frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
        frame["instrument"] = frame["instrument"].astype(str)
        return frame

    def _query_daily_bars(
        table, query_start, query_end, pool_start, pool_end
    ):
        # volume and amount are intraday cumulative fields in the official table;
        # their daily values are therefore the maximum/last cumulative snapshot.
        sql = f"""
        WITH minute_features AS (
            SELECT
                date,
                instrument,
                open,
                high,
                low,
                close,
                pre_close,
                volume,
                amount,
                CASE
                    WHEN bid_price1 > 0
                     AND ask_price1 >= bid_price1
                     AND (
                         COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0)
                         + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0)
                         + COALESCE(bid_volume5, 0)
                     ) > 0
                     AND (
                         COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0)
                         + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0)
                         + COALESCE(ask_volume5, 0)
                     ) > 0
                    THEN (
                        5.0 * COALESCE(bid_volume1, 0)
                        + 4.0 * COALESCE(bid_volume2, 0)
                        + 3.0 * COALESCE(bid_volume3, 0)
                        + 2.0 * COALESCE(bid_volume4, 0)
                        + COALESCE(bid_volume5, 0)
                        - 5.0 * COALESCE(ask_volume1, 0)
                        - 4.0 * COALESCE(ask_volume2, 0)
                        - 3.0 * COALESCE(ask_volume3, 0)
                        - 2.0 * COALESCE(ask_volume4, 0)
                        - COALESCE(ask_volume5, 0)
                    ) / (
                        5.0 * COALESCE(bid_volume1, 0)
                        + 4.0 * COALESCE(bid_volume2, 0)
                        + 3.0 * COALESCE(bid_volume3, 0)
                        + 2.0 * COALESCE(bid_volume4, 0)
                        + COALESCE(bid_volume5, 0)
                        + 5.0 * COALESCE(ask_volume1, 0)
                        + 4.0 * COALESCE(ask_volume2, 0)
                        + 3.0 * COALESCE(ask_volume3, 0)
                        + 2.0 * COALESCE(ask_volume4, 0)
                        + COALESCE(ask_volume5, 0)
                        + 1.0
                    )
                    ELSE NULL
                END AS book_imbalance,
                CASE
                    WHEN bid_price1 > 0 AND ask_price1 >= bid_price1
                    THEN (ask_price1 - bid_price1)
                         / (0.5 * (ask_price1 + bid_price1) + 1e-12)
                    ELSE NULL
                END AS relative_spread
            FROM {table}
            WHERE date >= CAST('{query_start}' AS DATETIME)
              AND date < CAST('{query_end}' AS DATETIME) + INTERVAL 1 DAY
              AND instrument IN (
                  SELECT instrument
                  FROM bigalpha_2026_instruments
                  WHERE date >= CAST('{pool_start}' AS DATETIME)
                    AND date < CAST('{pool_end}' AS DATETIME) + INTERVAL 1 DAY
                  GROUP BY instrument
              )
        )
        SELECT
            CAST(strftime(date, '%Y-%m-%d') AS DATETIME) AS date,
            instrument,
            ARG_MIN(open, date) AS open,
            MAX(high) AS high,
            MIN(low) AS low,
            ARG_MAX(close, date) AS close,
            ARG_MIN(pre_close, date) AS pre_close,
            MAX(volume) AS volume,
            MAX(amount) AS amount,
            AVG(
                CASE
                    WHEN strftime(date, '%H:%M:%S') >= '09:35:00'
                     AND strftime(date, '%H:%M:%S') <= '14:55:00'
                    THEN book_imbalance
                    ELSE NULL
                END
            ) AS day_book_imbalance,
            AVG(
                CASE
                    WHEN strftime(date, '%H:%M:%S') >= '14:30:00'
                     AND strftime(date, '%H:%M:%S') <= '14:55:00'
                    THEN book_imbalance
                    ELSE NULL
                END
            ) AS late_book_imbalance,
            quantile(
                CASE
                    WHEN strftime(date, '%H:%M:%S') >= '09:35:00'
                     AND strftime(date, '%H:%M:%S') <= '14:55:00'
                    THEN relative_spread
                    ELSE NULL
                END,
                0.5
            ) AS relative_spread,
            AVG(
                CASE
                    WHEN strftime(date, '%H:%M:%S') >= '09:35:00'
                     AND strftime(date, '%H:%M:%S') <= '14:55:00'
                    THEN CASE WHEN book_imbalance IS NOT NULL THEN 1.0 ELSE 0.0 END
                    ELSE NULL
                END
            ) AS book_coverage,
            AVG(
                CASE
                    WHEN strftime(date, '%H:%M:%S') >= '14:30:00'
                     AND strftime(date, '%H:%M:%S') <= '14:55:00'
                    THEN CASE WHEN book_imbalance IS NOT NULL THEN 1.0 ELSE 0.0 END
                    ELSE NULL
                END
            ) AS late_book_coverage
        FROM minute_features
        GROUP BY instrument, strftime(date, '%Y-%m-%d')
        """
        frame = dai.query(
            sql,
            filters={"date": [query_start, f"{query_end} 23:59:59"]},
            compression=True,
        ).df()
        return _normalise_date(frame)

    def _query_market_returns(table, query_start, query_end):
        # Build the market proxy from the actual constituent set on each date.
        # Unlike an end-date-wide instrument union, this definition is prefix
        # invariant when the requested prediction window is extended.
        sql = f"""
        WITH daily_prices AS (
            SELECT
                CAST(strftime(bar.date, '%Y-%m-%d') AS DATETIME) AS date,
                bar.instrument,
                ARG_MAX(bar.close, bar.date) AS close,
                ARG_MIN(bar.pre_close, bar.date) AS pre_close
            FROM {table} AS bar
            INNER JOIN (
                SELECT DISTINCT CAST(date AS DATETIME) AS date, instrument
                FROM bigalpha_2026_instruments
                WHERE date >= CAST('{query_start}' AS DATETIME)
                  AND date < CAST('{query_end}' AS DATETIME) + INTERVAL 1 DAY
            ) AS pool
              ON bar.instrument = pool.instrument
             AND CAST(strftime(bar.date, '%Y-%m-%d') AS DATETIME) = pool.date
            WHERE bar.date >= CAST('{query_start}' AS DATETIME)
              AND bar.date < CAST('{query_end}' AS DATETIME) + INTERVAL 1 DAY
            GROUP BY strftime(bar.date, '%Y-%m-%d'), bar.instrument
        )
        SELECT
            date,
            quantile(
                CASE
                    WHEN close > 0 AND pre_close > 0
                    THEN LN(close / pre_close)
                    ELSE NULL
                END,
                0.5
            ) AS market_return
        FROM daily_prices
        GROUP BY date
        """
        frame = dai.query(
            sql,
            filters={"date": [query_start, f"{query_end} 23:59:59"]},
            compression=True,
        ).df()
        frame = frame.copy()
        frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
        frame["market_return"] = pd.to_numeric(
            frame["market_return"], errors="coerce"
        ).astype("float32")
        return frame.drop_duplicates("date", keep="last").sort_values("date")

    def _query_financials(table, query_start, query_end):
        sql = f"""
        SELECT
            CAST(date AS DATETIME) AS date,
            instrument,
            operating_revenue,
            net_profit_to_parent_shareholders,
            total_assets,
            total_equity_to_parent_shareholders
        FROM {table}
        WHERE category = 'lf' AND shift = 0
          AND date >= CAST('{query_start}' AS DATETIME)
          AND date < CAST('{query_end}' AS DATETIME) + INTERVAL 1 DAY
        """
        frame = dai.query(
            sql,
            filters={"date": [query_start, f"{query_end} 23:59:59"]},
            compression=True,
        ).df()
        frame = _normalise_date(frame)
        value_columns = [
            "operating_revenue",
            "net_profit_to_parent_shareholders",
            "total_assets",
            "total_equity_to_parent_shareholders",
        ]
        for column in value_columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")

        # Keep one complete, real row.  Column-wise maxima can synthesize a
        # financial statement that never existed when duplicate rows disagree.
        frame["_nonnull"] = frame[value_columns].notna().sum(axis=1)
        frame = frame.sort_values(
            ["date", "instrument", "_nonnull"] + value_columns,
            kind="mergesort",
            na_position="first",
        )
        frame = (
            frame.drop_duplicates(["date", "instrument"], keep="last")
            .drop(columns="_nonnull")
            .reset_index(drop=True)
        )
        frame[value_columns] = frame[value_columns].astype("float32")
        return frame

    def _query_pool(query_start, query_end):
        pool = dai.query(
            f"""
            SELECT CAST(date AS DATETIME) AS date, instrument
            FROM bigalpha_2026_instruments
            WHERE date >= CAST('{query_start}' AS DATETIME)
              AND date < CAST('{query_end}' AS DATETIME) + INTERVAL 1 DAY
            """,
            filters={"date": [query_start, f"{query_end} 23:59:59"]},
            compression=True,
        ).df()
        return _normalise_date(pool).drop_duplicates(["date", "instrument"])

    def _safe_divide(numerator, denominator):
        denominator = denominator.where(denominator.abs() > 1e-12)
        return numerator / denominator

    def _build_features(bar_table, financial_table, period_start, period_end):
        period_start = pd.Timestamp(period_start).normalize()
        period_end = pd.Timestamp(period_end).normalize()
        # 260 calendar days safely covers the 120-trading-day momentum window.
        bar_start = (period_start - pd.Timedelta(days=260)).strftime("%Y-%m-%d")
        fin_start = (period_start - pd.Timedelta(days=730)).strftime("%Y-%m-%d")
        end_text = period_end.strftime("%Y-%m-%d")

        bars = _query_daily_bars(
            bar_table,
            bar_start,
            end_text,
            period_start.strftime("%Y-%m-%d"),
            end_text,
        )
        market_returns = _query_market_returns(bar_table, bar_start, end_text)
        financials = _query_financials(financial_table, fin_start, end_text)
        pool = _query_pool(period_start.strftime("%Y-%m-%d"), end_text)

        numeric_columns = [
            "open",
            "high",
            "low",
            "close",
            "pre_close",
            "volume",
            "amount",
            "day_book_imbalance",
            "late_book_imbalance",
            "relative_spread",
            "book_coverage",
            "late_book_coverage",
        ]
        for column in numeric_columns:
            bars[column] = pd.to_numeric(bars[column], errors="coerce").astype(
                "float32"
            )

        bars = bars.sort_values(["date", "instrument"]).reset_index(drop=True)
        financials = financials.rename(columns={"date": "financial_date"})
        financials = financials.sort_values(
            ["financial_date", "instrument"]
        ).reset_index(drop=True)
        data = pd.merge_asof(
            bars,
            financials,
            left_on="date",
            right_on="financial_date",
            by="instrument",
            direction="backward",
            allow_exact_matches=True,
        )
        data = data.merge(market_returns, on="date", how="left", validate="many_to_one")
        stale_financial = (data["date"] - data["financial_date"]).dt.days > 550
        financial_value_columns = [
            "operating_revenue",
            "net_profit_to_parent_shareholders",
            "total_assets",
            "total_equity_to_parent_shareholders",
        ]
        data.loc[stale_financial, financial_value_columns] = np.nan

        data = data.sort_values(["instrument", "date"]).reset_index(drop=True)
        data["core_valid"] = (
            (data["open"] > 0)
            & (data["close"] > 0)
            & (data["pre_close"] > 0)
            & (data["amount"] > 0)
        )
        valid_close = data["close"].where(data["core_valid"])
        valid_pre_close = data["pre_close"].where(data["core_valid"])
        valid_open = data["open"].where(data["core_valid"])
        close_to_preclose = _safe_divide(valid_close, valid_pre_close)
        data["ret_1d"] = np.log(close_to_preclose.where(close_to_preclose > 0))
        data["ret_1d"] = data["ret_1d"].clip(-0.30, 0.30)
        data["gap"] = _safe_divide(valid_open, valid_pre_close) - 1.0
        data["intraday_return"] = _safe_divide(valid_close, valid_open) - 1.0
        data["close_position"] = (
            _safe_divide(
                valid_close - data["low"].where(data["core_valid"]),
                data["high"].where(data["core_valid"])
                - data["low"].where(data["core_valid"]),
            )
            - 0.5
        )
        data["log_amount"] = np.log1p(
            data["amount"].where(data["core_valid"]).clip(lower=0)
        )

        instrument_key = data["instrument"]
        grouped = data.groupby("instrument", sort=False, group_keys=False)

        def _rolling(series, window, min_periods, operation):
            return series.groupby(instrument_key).transform(
                lambda values: getattr(
                    values.rolling(window, min_periods=min_periods), operation
                )()
            )

        valid_observation = data["core_valid"].astype(float).where(
            data["core_valid"]
        )
        data["history_count_120"] = _rolling(
            valid_observation, 120, 1, "count"
        )
        data["feature_ready"] = (
            data["core_valid"] & (data["history_count_120"] >= 120)
        )

        sum_3 = _rolling(data["ret_1d"], 3, 2, "sum")
        sum_20 = _rolling(data["ret_1d"], 20, 10, "sum")
        sum_120 = _rolling(data["ret_1d"], 120, 60, "sum")
        abs_sum_20 = _rolling(data["ret_1d"].abs(), 20, 10, "sum")
        abs_sum_120 = _rolling(data["ret_1d"].abs(), 120, 60, "sum")
        data["short_reversal_3"] = -sum_3
        data["momentum_120_excl_20"] = sum_120 - sum_20
        data["trend_efficiency_120_excl_20"] = _safe_divide(
            data["momentum_120_excl_20"], abs_sum_120 - abs_sum_20
        ).clip(-1.0, 1.0)

        data["volatility_20"] = _rolling(data["ret_1d"], 20, 10, "std")
        volatility_60 = _rolling(data["ret_1d"], 60, 30, "std")
        downside_return = data["ret_1d"].where(data["ret_1d"] < 0, 0.0)
        data["downside_volatility_20"] = _rolling(
            downside_return, 20, 10, "std"
        )
        data["volatility_ratio_20_60"] = _safe_divide(
            data["volatility_20"], volatility_60
        ).clip(0.0, 5.0)

        data["log_amount_20"] = grouped["log_amount"].transform(
            lambda values: values.rolling(20, min_periods=10).mean()
        )
        data["amount_surprise_20"] = data["log_amount"] - data["log_amount_20"]
        amihud_raw = _safe_divide(data["ret_1d"].abs(), data["amount"] + 1.0)
        data["amihud_20"] = np.log1p(
            1e8 * _rolling(amihud_raw, 20, 10, "median").clip(lower=0)
        )

        data["market_return"] = data["market_return"].clip(-0.20, 0.20)
        data["return_x_market"] = data["ret_1d"] * data["market_return"]
        data["market_return_sq"] = data["market_return"] ** 2
        rolling_mean_60 = lambda values: values.rolling(60, min_periods=30).mean()
        mean_return_60 = data["ret_1d"].groupby(instrument_key).transform(
            rolling_mean_60
        )
        mean_market_60 = data["market_return"].groupby(instrument_key).transform(
            rolling_mean_60
        )
        mean_cross_60 = grouped["return_x_market"].transform(rolling_mean_60)
        mean_market_sq_60 = grouped["market_return_sq"].transform(rolling_mean_60)
        market_variance_60 = mean_market_sq_60 - mean_market_60**2
        data["beta_60"] = _safe_divide(
            mean_cross_60 - mean_return_60 * mean_market_60,
            market_variance_60,
        ).clip(-5.0, 5.0)
        residual_return = data["ret_1d"] - data["beta_60"] * data["market_return"]
        data["residual_volatility_20"] = _rolling(
            residual_return, 20, 10, "std"
        )

        data["late_book_imbalance"] = data["late_book_imbalance"].where(
            data["core_valid"] & (data["late_book_coverage"] >= 0.50)
        )
        data["day_book_imbalance"] = data["day_book_imbalance"].where(
            data["core_valid"] & (data["book_coverage"] >= 0.50)
        )
        data["relative_spread"] = data["relative_spread"].where(
            data["core_valid"]
            & (data["book_coverage"] >= 0.50)
            & (data["relative_spread"] >= 0)
            & (data["relative_spread"] <= 0.10)
        )
        data["book_imbalance_change"] = (
            data["late_book_imbalance"] - data["day_book_imbalance"]
        )

        profit = data["net_profit_to_parent_shareholders"]
        revenue = data["operating_revenue"]
        assets = data["total_assets"]
        equity = data["total_equity_to_parent_shareholders"]
        data["roa_proxy"] = _safe_divide(profit, assets.where(assets > 0))
        data["net_margin"] = _safe_divide(profit, revenue.abs().where(revenue != 0))
        data["asset_turnover"] = _safe_divide(revenue, assets.where(assets > 0))
        data["leverage"] = _safe_divide(
            assets, equity.where((assets > 0) & (equity > 0))
        )

        data = data[data["date"].between(period_start, period_end)].copy()
        data.replace([np.inf, -np.inf], np.nan, inplace=True)
        label_rows = data[
            ["date", "instrument", "ret_1d", "core_valid"]
        ].copy()
        data = pool.merge(data, on=["date", "instrument"], how="left")
        data["core_valid"] = data["core_valid"].fillna(False).astype(bool)
        data["feature_ready"] = data["feature_ready"].fillna(False).astype(bool)
        return_columns = list(
            dict.fromkeys(
                [
                    "date",
                    "instrument",
                    "core_valid",
                    "feature_ready",
                    "ret_1d",
                ]
                + rank_input_columns
            )
        )
        float_columns = [
            column
            for column in return_columns
            if column
            not in {"date", "instrument", "core_valid", "feature_ready"}
        ]
        data[float_columns] = data[float_columns].astype("float32")
        feature_rows = (
            data[return_columns]
            .sort_values(["date", "instrument"])
            .reset_index(drop=True)
        )
        label_rows["ret_1d"] = label_rows["ret_1d"].astype("float32")
        return feature_rows, label_rows.sort_values(
            ["date", "instrument"]
        ).reset_index(drop=True)

    def _rank_features(frame):
        ranked = frame.copy()
        for column in rank_input_columns:
            valid_values = ranked[column].where(ranked["feature_ready"])
            ranked[column] = (
                valid_values.groupby(ranked["date"])
                .rank(method="average", pct=True)
                - 0.5
            )

        quality_parts = pd.concat(
            [
                ranked["roa_proxy"],
                ranked["net_margin"],
                ranked["asset_turnover"],
                -ranked["leverage"],
            ],
            axis=1,
        )
        quality_available = quality_parts.notna().sum(axis=1) >= 2
        quality_raw = quality_parts.mean(axis=1, skipna=True).where(
            ranked["feature_ready"] & quality_available
        )
        ranked["quality_score"] = (
            quality_raw.groupby(ranked["date"]).rank(method="average", pct=True)
            - 0.5
        )
        ranked[rank_input_columns] = ranked[rank_input_columns].fillna(0.0)
        ranked["quality_score"] = ranked["quality_score"].fillna(0.0)
        ranked["confirmed_reversal"] = ranked["short_reversal_3"] * (
            0.5 + ranked["amount_surprise_20"]
        )
        ranked["low_vol_momentum"] = ranked["momentum_120_excl_20"] * (
            0.5 - ranked["residual_volatility_20"]
        )
        ranked["book_flow_quality"] = ranked["late_book_imbalance"] * (
            0.5 - ranked["relative_spread"]
        )
        ranked["quality_momentum"] = (
            ranked["quality_score"] * ranked["momentum_120_excl_20"]
        )
        prepared_columns = list(dict.fromkeys(model_columns + style_control_columns))
        ranked[prepared_columns] = ranked[prepared_columns].fillna(0.0)
        ranked[prepared_columns] = ranked[prepared_columns].astype("float32")
        return ranked

    def _residualize_by_date(frame, value_column, control_columns):
        residual = pd.Series(np.nan, index=frame.index, dtype=float)
        for _, indices in frame.groupby("date", sort=False).groups.items():
            y = frame.loc[indices, value_column].to_numpy(dtype=float)
            z = frame.loc[indices, control_columns].to_numpy(dtype=float)
            valid = np.isfinite(y) & np.isfinite(z).all(axis=1)
            if valid.sum() < max(30, 5 * (len(control_columns) + 1)):
                residual.loc[indices] = y
                continue
            design = np.column_stack([np.ones(valid.sum()), z[valid]])
            gram = design.T @ design
            penalty = 1e-3 * valid.sum()
            gram[1:, 1:] += penalty * np.eye(len(control_columns))
            rhs = design.T @ y[valid]
            try:
                coefficients = np.linalg.solve(gram, rhs)
            except np.linalg.LinAlgError:
                coefficients = np.linalg.lstsq(gram, rhs, rcond=None)[0]
            values = y.copy()
            values[valid] = y[valid] - design @ coefficients
            residual.loc[indices] = values
        return residual

    def _fit_weighted_ridge(features, target, sample_weight):
        x = features.to_numpy(dtype=float)
        y = target.to_numpy(dtype=float)
        w = np.asarray(sample_weight, dtype=float)
        design = np.column_stack([np.ones(len(x)), x])
        gram = design.T @ (w[:, None] * design)
        penalty = 3e-3 * w.sum()
        gram[1:, 1:] += penalty * np.eye(x.shape[1])
        rhs = design.T @ (w * y)
        try:
            return np.linalg.solve(gram, rhs)
        except np.linalg.LinAlgError:
            return np.linalg.lstsq(gram, rhs, rcond=None)[0]

    def _ridge_predict(features, coefficients):
        x = features.to_numpy(dtype=float)
        return coefficients[0] + x @ coefficients[1:]

    def _prepare_training_chunk(chunk_start_ts, chunk_end_ts):
        # Include a small forward calendar overlap only to obtain the next
        # trading-day label for the last feature date in a non-final chunk.
        # The overlap is clipped at the fixed official training end, so no
        # evaluation-period return can enter training.
        label_end_ts = min(chunk_end_ts + pd.Timedelta(days=20), train_end_ts)
        raw, label_rows = _build_features(
            train_bar_table,
            train_financial_table,
            chunk_start_ts.strftime("%Y-%m-%d"),
            label_end_ts.strftime("%Y-%m-%d"),
        )
        ranked = _rank_features(raw)

        # Map each realised return date to the immediately preceding global
        # trading date. A per-stock shift can jump across months when a
        # constituent exits and later re-enters the pool.
        valid_calendar = label_rows.groupby("date")["core_valid"].sum()
        calendar_minimum = min(
            300, int(np.ceil(0.70 * valid_calendar.max()))
        )
        calendar = pd.DataFrame(
            {
                "realized_date": valid_calendar[
                    valid_calendar >= calendar_minimum
                ].index.sort_values()
            }
        )
        calendar["date"] = calendar["realized_date"].shift(1)
        labels = label_rows[
            ["date", "instrument", "ret_1d", "core_valid"]
        ].rename(
            columns={
                "date": "realized_date",
                "ret_1d": "target",
                "core_valid": "target_valid",
            }
        )
        labels = labels.merge(calendar, on="realized_date", how="inner")
        labels = labels[labels["target_valid"]].dropna(
            subset=["date", "target"]
        )

        features = ranked[
            ranked["date"].between(chunk_start_ts, chunk_end_ts)
        ]
        chunk = features.merge(
            labels[["date", "instrument", "target"]],
            on=["date", "instrument"],
            how="inner",
            validate="one_to_one",
        )
        pool_count = features.groupby("date")["instrument"].size()
        chunk = chunk[chunk["feature_ready"]].copy()
        usable_count = chunk.groupby("date")["instrument"].size()
        coverage = usable_count.div(pool_count).fillna(0.0)
        minimum_usable = pd.Series(
            np.minimum(300, np.ceil(0.70 * pool_count)),
            index=pool_count.index,
        ).reindex(usable_count.index)
        good_dates = coverage[
            (usable_count >= minimum_usable) & (coverage >= 0.70)
        ].index
        chunk = chunk[chunk["date"].isin(good_dates)].copy()
        chunk["target"] = chunk.groupby("date")["target"].rank(pct=True) - 0.5
        chunk["target_neutral"] = _residualize_by_date(
            chunk, "target", style_control_columns
        )
        chunk["target_neutral"] = (
            chunk.groupby("date")["target_neutral"].rank(pct=True) - 0.5
        )
        chunk = chunk.dropna(subset=["target_neutral"])

        # A deterministic, date-stratified half-universe sample is enough for
        # the smooth target while keeping the 8 GiB AI Studio runner stable.
        # Ranking and style neutralisation are still computed on the full daily
        # cross-section before sampling.
        instrument_number = pd.to_numeric(
            chunk["instrument"].str.extract(r"(\d+)", expand=False),
            errors="coerce",
        ).fillna(0).astype("uint64")
        exchange_number = (
            chunk["instrument"].str.endswith(".SH").astype("uint64")
            + 2 * chunk["instrument"].str.endswith(".SZ").astype("uint64")
            + 3 * chunk["instrument"].str.endswith(".BJ").astype("uint64")
        )
        date_number = (
            chunk["date"].astype("int64") // 86_400_000_000_000
        ).astype("uint64")
        chunk["_sample_hash"] = (
            instrument_number * np.uint64(11_400_714_819_323_198_485)
        ) ^ (date_number * np.uint64(14_029_467_366_897_019_727)) ^ (
            exchange_number * np.uint64(1_609_587_929_392_839_161)
        )
        chunk = (
            chunk.sort_values(
                ["date", "_sample_hash", "instrument"], kind="mergesort"
            )
            .groupby("date", sort=False, group_keys=False)
            .head(train_rows_per_date)
        )
        keep_columns = ["date"] + model_columns + ["target_neutral"]
        chunk = chunk[keep_columns].reset_index(drop=True)
        chunk[model_columns + ["target_neutral"]] = chunk[
            model_columns + ["target_neutral"]
        ].astype("float32")
        return chunk

    training_parts = []
    chunk_start_ts = official_history_start
    while chunk_start_ts <= train_end_ts:
        chunk_end_ts = min(
            chunk_start_ts + pd.Timedelta(days=training_chunk_days - 1),
            train_end_ts,
        )
        part = _prepare_training_chunk(chunk_start_ts, chunk_end_ts)
        if not part.empty:
            training_parts.append(part)
        del part
        gc.collect()
        chunk_start_ts = chunk_end_ts + pd.Timedelta(days=1)

    if not training_parts:
        raise ValueError("Insufficient fixed-history training observations.")
    train = pd.concat(training_parts, ignore_index=True)
    del training_parts
    gc.collect()
    if len(train) < 1000:
        raise ValueError("Insufficient fixed-history training observations.")

    age_days = (train_end_ts - train["date"]).dt.days.clip(lower=0)
    time_weight = 0.25 + 0.75 * np.power(0.5, age_days / 730.0)
    daily_count = train.groupby("date")["date"].transform("size")
    sample_weight = time_weight / daily_count.clip(lower=1)
    sample_weight = sample_weight / sample_weight.mean()

    ridge_coefficients = _fit_weighted_ridge(
        train[model_columns], train["target_neutral"], sample_weight
    )
    xgb_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=140,
        max_depth=2,
        learning_rate=0.025,
        min_child_weight=300,
        gamma=0.02,
        subsample=1.0,
        colsample_bytree=1.0,
        reg_alpha=1.0,
        reg_lambda=20.0,
        max_bin=64,
        tree_method="hist",
        n_jobs=1,
        random_state=42,
        verbosity=0,
    )
    xgb_model.fit(
        train[model_columns],
        train["target_neutral"],
        sample_weight=sample_weight,
    )
    del train, sample_weight
    gc.collect()

    test_raw, _ = _build_features(
        datasources["bar1m"], datasources["financial"], start_date, end_date
    )
    test = _rank_features(test_raw)
    valid_test = test["feature_ready"] & np.isfinite(
        test[model_columns].to_numpy(dtype="float32", copy=False)
    ).all(axis=1)
    scoring_columns = list(dict.fromkeys(model_columns + style_control_columns))
    scored = test.loc[
        valid_test, ["date", "instrument"] + scoring_columns
    ].copy()
    # A deliberately truncated self-test datasource may not contain the
    # 120 observations required by feature_ready.  In that valid edge case,
    # return the complete pool with neutral factors instead of asking pandas
    # or XGBoost to rank/predict an empty frame.
    if not scored.empty:
        scored["ridge_prediction"] = _ridge_predict(
            scored[model_columns], ridge_coefficients
        )
        scored["xgb_prediction"] = xgb_model.predict(scored[model_columns])
        scored["ridge_rank"] = (
            scored.groupby("date")["ridge_prediction"].rank(pct=True) - 0.5
        )
        scored["xgb_rank"] = (
            scored.groupby("date")["xgb_prediction"].rank(pct=True) - 0.5
        )
        scored["blend"] = 0.70 * scored["ridge_rank"] + 0.30 * scored["xgb_rank"]
        scored["factor"] = (
            scored.groupby("date")["blend"].rank(pct=True) - 0.5
        )

    result = test[["date", "instrument"]].copy()
    result["factor"] = 0.0
    if not scored.empty:
        result.loc[scored.index, "factor"] = scored["factor"].to_numpy()
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce")
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    result = result.drop_duplicates(["date", "instrument"], keep="last")
    return result.sort_values(["date", "instrument"]).reset_index(drop=True)
